In [1]:
import os

os.environ["TORCHDYNAMO_DISABLE"] = "1"

import torch
import warnings
import logging


from dftorch import Constants, Structure, ESDriver


# Silence warnings and torch logs
warnings.filterwarnings("ignore")
os.environ["TORCH_LOGS"] = ""
os.environ["TORCHINDUCTOR_VERBOSE"] = "0"
os.environ["TORCHDYNAMO_VERBOSE"] = "0"
logging.getLogger("torch.fx").setLevel(logging.CRITICAL)
logging.getLogger("torch.fx.experimental.symbolic_shapes").setLevel(logging.CRITICAL)
logging.getLogger("torch.fx.experimental.recording").setLevel(logging.CRITICAL)

torch._dynamo.config.capture_dynamic_output_shape_ops = True
torch.set_default_dtype(torch.float64)
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


# Vibrational mode calculation.
### Done via finite differences. Displacements are batched.
### Solvation model introduces a significant overhead, so there is 'frozen_gbsa' option which uses frozen geometry but updated upon displacement charges.

In [3]:
%%time
dftorch_params = {
    "FILENAME": "COORD.xyz",
    "SKFPATH": "sk_orig/3ob-3-1/",  # Path to SKF files
    "T_ELECTRONIC": 1000.0,  # Electronic temperature in Kelvin for Fermi smearing
    "RCUT_ELECTRONIC": 7.0,  # Cutoff for electronic interactions in Angstroms. Should be >= largest cutoff in SKF files for the element pairs present in the system.
    "RCUT_REPULSIVE": 4.0,  # Cutoff for repulsive interactions in Angstroms. Should be >= largest cutoff in SKF files for the element pairs present in the system.
    "COUL_METHOD": "FULL",  # 'FULL' for full coulomb matrix, 'PME' for Particle Mesh Ewald method

    "SCF_TOL": 1e-7,
    "SCF_ALPHA": 0.05,
    "KRYLOV_START": 20,
    "D3_PARAMS": {"s6": 1.0, "s8": 0.5883, "a1": 0.5719, "a2": 3.6107},
    "SOLVENT_PARAM_FILE": "sk_orig/3ob-3-1/solvation/param_gbsa_h2o.txt",
    "SOLVATION_MODEL": "gbsa",
    "H_DAMP_EXP": 4.0,
}

filename = "COORD_8WATER.xyz"

const = Constants(dftorch_params).to(device)

structure1 = Structure(dftorch_params, const, device=device)

es_driver = ESDriver(dftorch_params, device=device)

es_driver(structure1, const, do_scf=True)
es_driver.calc_forces(structure1, const)

Ha = 0.0367493  # correct Ha → eV conversion
# print(f"  {'net spin':22s} {structure1.net_spin_sr.sum().item():12.4f}")
print(f"{'Energy Components':─^52}")
print(
    f"  {'E_total':22s} {structure1.e_tot * Ha:12.6f} Ha  ({structure1.e_tot:12.6f} eV)"
)
print(
    f"  {'E_band':22s} {structure1.e_band0 * Ha:12.6f} Ha ({structure1.e_band0:12.6f} eV)"
)
print(
    f"  {'E_coulomb':22s} {structure1.e_coul * Ha:12.6f} Ha ({structure1.e_coul:12.6f} eV)"
)
print(
    f"  {'E_repulsion':22s} {structure1.e_repulsion * Ha:12.6f} Ha ({structure1.e_repulsion:12.6f} eV)"
)
print(
    f"  {'E_entropy (-TS)':22s} {structure1.e_entropy * Ha:12.6f} Ha  ({structure1.e_entropy:12.6f} eV)"
)
print(
    f"  {'E_solvation':22s} {structure1.e_solv * Ha:12.6f} Ha  ({structure1.e_solv:12.6f} eV)"
)
print(f"  {'E_d3':22s} {structure1.e_d3 * Ha:12.6f} Ha  ({structure1.e_d3:12.6f} eV)")
# print(
#     f"  {'E_spin':22s} {structure1.e_spin.item() * Ha:12.6f} Ha  ({structure1.e_spin.item():12.6f} eV)"
# )
print(f"{'─' * 52}")
print(
    f"  {'E_free (F=E-TS)':22s} {(structure1.e_tot - structure1.e_entropy) * Ha:12.6f} Ha ({(structure1.e_tot - structure1.e_entropy):12.6f} eV)"
)

DFTB3: False
coulomb_matrix_vectorized
  Coulomb_Real t 0.0 s
### Do _scf ###
  Initial dm_fermi
  Initial mu = 0.2598

Starting cycle
Iter 1
Res = 0.227886536, dEc = 0.145746506, t = 0.0 s

Iter 2
Res = 0.189989605, dEc = 0.127869487, t = 0.0 s

Iter 3
Res = 0.022378231, dEc = 0.003270510, t = 0.0 s

Iter 4
Res = 0.000899906, dEc = 0.000273368, t = 0.0 s

Iter 5
Res = 0.000666270, dEc = 0.000272063, t = 0.0 s

Iter 6
Res = 0.000129966, dEc = 0.000041092, t = 0.0 s

Iter 7
Res = 0.000011996, dEc = 0.000000982, t = 0.0 s

Iter 8
Res = 0.000001500, dEc = 0.000000274, t = 0.0 s

Iter 9
Res = 0.000000647, dEc = 0.000000053, t = 0.0 s

Iter 10
Res = 0.000000449, dEc = 0.000000021, t = 0.0 s

Iter 11
Res = 0.000000066, dEc = 0.000000005, t = 0.0 s

─────────────────Energy Components──────────────────
  E_total                   -2.861174 Ha  (  -77.856570 eV)
  E_band                    -2.482861 Ha (  -67.562134 eV)
  E_coulomb                 -0.009937 Ha (   -0.270389 eV)
  E_repulsion   

In [4]:
es_driver.calc_hessian(
    structure1,
    const,
    mode="full",  # "full" | "frozen_gbsa"
    batch_size=64,
    verbose=True,
);

FD Hessian (full): 20 atoms, 60 DOFs, δ = 0.0001 Å
  batch_size=64 → 32 DOFs/batch, 2 batched ES calls
GBSA initialization time: 2.58 seconds
  Using q_init (skipping initial dm_fermi)

Starting cycle
Iter 1
Batch 0: Res = 8.615e-05, dEc = 2.704e-01
Batch 1: Res = 8.613e-05, dEc = 2.704e-01
Batch 2: Res = 1.668e-05, dEc = 2.704e-01
Batch 3: Res = 1.670e-05, dEc = 2.704e-01
Batch 4: Res = 4.906e-05, dEc = 2.704e-01
Batch 5: Res = 4.902e-05, dEc = 2.704e-01
Batch 6: Res = 2.202e-05, dEc = 2.704e-01
Batch 7: Res = 2.200e-05, dEc = 2.704e-01
Batch 8: Res = 2.234e-05, dEc = 2.704e-01
Batch 9: Res = 2.235e-05, dEc = 2.704e-01
Batch 10: Res = 2.655e-05, dEc = 2.704e-01
Batch 11: Res = 2.656e-05, dEc = 2.704e-01
Batch 12: Res = 1.370e-05, dEc = 2.704e-01
Batch 13: Res = 1.370e-05, dEc = 2.704e-01
Batch 14: Res = 4.221e-05, dEc = 2.704e-01
Batch 15: Res = 4.220e-05, dEc = 2.704e-01
Batch 16: Res = 1.745e-05, dEc = 2.704e-01
Batch 17: Res = 1.747e-05, dEc = 2.704e-01
Batch 18: Res = 6.591e-05, d

In [5]:
freqs_cm, eigvecs, results = ESDriver.calc_frequencies(structure1, proj_tr=True)

N_at = structure1.Nats
n3 = 3 * N_at

print(f"{'Vibrational Frequencies':─^60}")
print(f"  N_atoms = {N_at},  3N = {n3},  6 T+R modes expected\n")
print(f"  {'Mode':>5s}  {'Eigenvalue':>14s}  {'Freq (cm⁻¹)':>14s}  {'Note':>8s}")
print(f"  {'─' * 50}")
for k in range(n3):
    note = ""
    if abs(freqs_cm[k]) < 10:
        note = "T/R"
    elif freqs_cm[k] < -10:
        note = "imag"
    print(
        f"  {k + 1:5d}  {results['eigenvalues'][k]:14.6f}  {freqs_cm[k]:14.2f}  {note:>8s}"
    )

print(
    f"\n  Conversion factor: {results['conversion_factor']:.3f} cm⁻¹ / sqrt(eV/(Å²·amu))"
)
print(f"  6 lowest freqs: {freqs_cm[:6]}")
print(f"  Highest freq:   {freqs_cm[-1]:.2f} cm⁻¹")
print(f"  Imaginary modes (< -10 cm⁻¹): {results['n_imag']}")

──────────────────Vibrational Frequencies───────────────────
  N_atoms = 20,  3N = 60,  6 T+R modes expected

   Mode      Eigenvalue     Freq (cm⁻¹)      Note
  ──────────────────────────────────────────────────
      1       -2.331846         -796.31      imag
      2       -1.925415         -723.59      imag
      3       -1.408886         -618.97      imag
      4       -0.964474         -512.12      imag
      5       -0.637889         -416.49      imag
      6       -0.362475         -313.96      imag
      7       -0.000000           -0.00       T/R
      8       -0.000000           -0.00       T/R
      9        0.000000            0.00       T/R
     10        0.000000            0.00       T/R
     11        0.000000            0.00       T/R
     12        0.000000            0.00       T/R
     13        0.350051          308.53          
     14        0.667259          425.97          
     15        0.772318          458.28          
     16        1.315029          598.